# Aula 04 — Probabilidade e Distribuições

## Objetivos
1. Entender conceitos fundamentais de probabilidade.
2. Conhecer as principais **distribuições discretas**: Bernoulli, Binomial, Poisson.
3. Conhecer as principais **distribuições contínuas**: Uniforme, Exponencial.
4. Simular e visualizar distribuições com `scipy.stats`.

---

## 1. Conceitos fundamentais

### Experimento aleatório
Procedimento cujo resultado não pode ser previsto com certeza (lançar um dado, sortear um cidadão para a PNAD).

### Espaço amostral ($\Omega$)
Conjunto de todos os resultados possíveis. Ex.: $\Omega = \{1, 2, 3, 4, 5, 6\}$ para um dado.

### Evento
Subconjunto de $\Omega$. Ex.: "sair número par" = $\{2, 4, 6\}$.

### Probabilidade (axiomas de Kolmogorov)
1. $P(A) \ge 0$
2. $P(\Omega) = 1$
3. Se $A_1, A_2, \ldots$ disjuntos: $P(\bigcup A_i) = \sum P(A_i)$

### Regras essenciais
- **Complementar:** $P(A^c) = 1 - P(A)$
- **União:** $P(A \cup B) = P(A) + P(B) - P(A \cap B)$
- **Condicional:** $P(A|B) = \dfrac{P(A \cap B)}{P(B)}$
- **Independência:** $P(A \cap B) = P(A) \cdot P(B)$

---

## 2. Variável aleatória

Função que associa números aos resultados de um experimento. Pode ser:
- **Discreta:** valores enumeráveis (0, 1, 2, ...).
- **Contínua:** valores em um intervalo de $\mathbb{R}$.

### Função de massa (PMF) — discreta
$$p(x) = P(X = x)$$

### Função densidade (PDF) — contínua
$$f(x) \ge 0, \quad \int_{-\infty}^{\infty} f(x)\,dx = 1$$
$$P(a < X < b) = \int_a^b f(x)\,dx$$

### Função de distribuição acumulada (CDF)
$$F(x) = P(X \le x)$$

---

## 3. Distribuições Discretas

### 3.1 Bernoulli($p$)
Único ensaio com 2 resultados: sucesso (1) com prob $p$, fracasso (0) com prob $1-p$.
$$E[X] = p, \quad \mathrm{Var}(X) = p(1-p)$$

### 3.2 Binomial($n, p$)
Número de sucessos em $n$ ensaios Bernoulli independentes.
$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$$
$$E[X] = np, \quad \mathrm{Var}(X) = np(1-p)$$

### 3.3 Poisson($\lambda$)
Número de eventos em intervalo fixo (tempo/espaço) com taxa média $\lambda$.
$$P(X = k) = \frac{e^{-\lambda} \lambda^k}{k!}$$
$$E[X] = \mathrm{Var}(X) = \lambda$$

---

## 4. Distribuições Contínuas

### 4.1 Uniforme($a, b$)
$$f(x) = \frac{1}{b-a}, \quad a \le x \le b$$

### 4.2 Exponencial($\lambda$)
Tempo entre eventos em processo de Poisson.
$$f(x) = \lambda e^{-\lambda x}, \quad x \ge 0$$
$$E[X] = 1/\lambda$$

In [ ]:
# === SETUP PARA GOOGLE COLAB ===
# Esta célula garante que o notebook funcione tanto em ambiente local
# quanto em quem clonar o repositório direto no Colab.

import sys, os, subprocess

# 1) Instala dependências (silencioso se já existirem)
pkgs = ["numpy", "pandas", "matplotlib", "seaborn", "scipy", "requests", "plotly", "statsmodels"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# 2) Se estivermos no Colab e o repo ainda não foi clonado, clona.
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/220719/curso-estatistica-python.git"  # ajuste após criar o repo

if IN_COLAB and not os.path.exists("/content/curso-estatistica-python"):
    subprocess.run(["git", "clone", REPO_URL, "/content/curso-estatistica-python"], check=False)

# 3) Adiciona o diretório utils ao PYTHONPATH para importar o módulo SIDRA
BASE = "/content/curso-estatistica-python" if IN_COLAB else ".."
if BASE not in sys.path:
    sys.path.append(BASE)

# 4) Pasta de gráficos (cria se não existir)
GRAFICOS_DIR = os.path.join(BASE, "graficos")
os.makedirs(GRAFICOS_DIR, exist_ok=True)

print("✓ Ambiente pronto. Pasta de gráficos:", GRAFICOS_DIR)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)  # gerador reprodutível

## 5. Simulação: Binomial

Imagine pesquisar 100 brasileiros para saber se moram em domicílio com saneamento básico.
Suponha que a verdadeira proporção seja $p = 0{,}68$.

Pergunta: qual a probabilidade de obtermos **exatamente 70** com saneamento na amostra?

In [ ]:
n, p = 100, 0.68
X = stats.binom(n, p)

# P(X = 70)
print(f"P(X = 70) = {X.pmf(70):.4f}")
# P(X >= 75)
print(f"P(X ≥ 75) = {1 - X.cdf(74):.4f}")
# Média e variância teóricas
print(f"E[X] = {X.mean()}, Var(X) = {X.var()}")

In [ ]:
# Visualizando a PMF
k = np.arange(50, 90)
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(k, X.pmf(k), color="steelblue", edgecolor="white")
ax.axvline(X.mean(), color="red", linestyle="--", label=f"E[X] = {X.mean():.0f}")
ax.set_title(f"Distribuição Binomial(n={n}, p={p})", fontweight="bold")
ax.set_xlabel("Número de sucessos (k)")
ax.set_ylabel("P(X = k)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, "aula04_binomial_pmf.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Simulação: Poisson com dados reais

O IBGE divulga estatísticas vitais. Vamos **modelar** o número de nascimentos diários
em um município hipotético com média $\lambda = 12$ nascimentos/dia.

> Em sala você pode trocar por dados reais do SIM/SINASC (DataSUS).

In [ ]:
lam = 12
Y = stats.poisson(lam)

# Simulamos 365 dias de um ano
amostras = Y.rvs(size=365, random_state=rng)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(amostras, bins=range(0, 30), density=True, alpha=0.6,
        color="orange", edgecolor="white", label="Simulação (365 dias)")
k = np.arange(0, 30)
ax.plot(k, Y.pmf(k), "o-", color="darkred", label="PMF teórica")
ax.set_title(f"Poisson(λ={lam}) — Nascimentos diários simulados", fontweight="bold")
ax.set_xlabel("Nascimentos no dia")
ax.set_ylabel("Frequência relativa")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, "aula04_poisson.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"Média simulada: {amostras.mean():.2f}  (teórica: {lam})")
print(f"Variância simulada: {amostras.var():.2f}  (teórica: {lam})")

## 7. Aplicação ao IPCA: estimando probabilidade empírica

Sem assumir distribuição, podemos calcular **probabilidades empíricas**
diretamente dos dados.

In [ ]:
from utils.sidra_api import obter_ipca_mensal
df_ipca = obter_ipca_mensal(120)
ipca = df_ipca["variacao_mensal"]

# P(IPCA > 1% no mês) — empírica
p_alto = (ipca > 1.0).mean()
# P(deflação) — empírica
p_def = (ipca < 0).mean()

print(f"P(IPCA mensal > 1%)  ≈ {p_alto:.3f}")
print(f"P(deflação mensal)   ≈ {p_def:.3f}")

## 8. Exponencial: tempo entre eventos raros

Se eventos seguem Poisson($\lambda$), o **tempo entre eventos** segue Exponencial($\lambda$).

In [ ]:
lam_exp = 0.5  # 0.5 eventos por hora → 1 evento a cada 2h em média
Z = stats.expon(scale=1/lam_exp)

x = np.linspace(0, 12, 200)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, Z.pdf(x), color="purple", linewidth=2, label="PDF")
ax.fill_between(x[x<=2], Z.pdf(x[x<=2]), alpha=0.3, color="purple",
                label=f"P(X≤2) = {Z.cdf(2):.3f}")
ax.set_title(f"Exponencial(λ={lam_exp})", fontweight="bold")
ax.set_xlabel("Tempo (horas)")
ax.set_ylabel("Densidade")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, "aula04_exponencial.png"), dpi=150, bbox_inches="tight")
plt.show()

## Exercícios

1. Numa eleição com $p=0{,}55$ de vitória, qual a probabilidade de em 1000 entrevistados,
   pelo menos 580 declararem voto no candidato?
2. Se o atendimento de um posto de saúde tem média de 8 pacientes/hora (Poisson),
   qual a probabilidade de receber 12 ou mais pacientes em uma hora?
3. Calcule a probabilidade empírica de **dois meses consecutivos** com IPCA negativo
   nos últimos 10 anos.

**Próxima aula:** Distribuição Normal e Teorema Central do Limite.